In [19]:
%run_nb spark-start --data-format iceberg

Args: Namespace(data_format='iceberg') - unknown_args: []
Spark version: 4.1.3, Driver memory: 16g, Executor memory: 8g, Service: jupyter-spark-4.1, Data format: iceberg
Port offset: 3
Spark packages: org.postgresql:postgresql:42.7.7,org.apache.hadoop:hadoop-aws:3.4.2,org.apache.spark:spark-avro_2.13:4.1.3,org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.3,org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.11.0
Spark extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Spark catalog configs: {'spark.sql.catalog.local': 'org.apache.iceberg.spark.SparkCatalog', 'spark.sql.catalog.local.type': 'hadoop', 'spark.sql.catalog.local.warehouse': 'file:///home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse'}


Loading ITables v2.9.1 from the internet... (need help?)
🔒ⓘcatalog
local
spark_catalog


Loading ITables v2.9.1 from the internet... (need help?)
🔒ⓘnamespace
default


Catalog local: org.apache.iceberg.spark.SparkCatalog. Current: 
org.apache.spark.sql.execution.datasources.v2.V2SessionCatalog


Version,4.1.3
Master,local[12]
AppName,main


               total        used        free      shared  buff/cache   available
Mem:            62Gi        32Gi       1.6Gi       346Mi        29Gi        29Gi
Swap:          8.0Gi          0B       8.0Gi
time: 150 ms (started: 2026-09-04 12:35:00 +00:00)


In [20]:
path="/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset"
!ls -la {path}
pd.set_option("display.max_colwidth", None) # Disable collumn truncation

total 5799848
drwxr-xr-x. 1 jovyan users        332 Aug 18 20:25  .
drwxr-xr-x. 1 jovyan users         62 Aug 18 20:24  ..
drwxr-xr-x. 1 jovyan users         16 Aug 18 20:25  .complete
-rw-r--r--. 1 jovyan users       5356 Aug 18 20:25  female_coaches.csv
-rw-r--r--. 1 jovyan users    1685124 Aug 18 20:25 'female_players (legacy).csv'
-rw-r--r--. 1 jovyan users   94212088 Aug 18 20:25  female_players.csv
-rw-r--r--. 1 jovyan users    2214250 Aug 18 20:25  female_teams.csv
-rw-r--r--. 1 jovyan users     132879 Aug 18 20:25  male_coaches.csv
-rw-r--r--. 1 jovyan users   90933390 Aug 18 20:25 'male_players (legacy).csv'
-rw-r--r--. 1 jovyan users 5637100640 Aug 18 20:25  male_players.csv
-rw-r--r--. 1 jovyan users  112744779 Aug 18 20:25  male_teams.csv
drwxr-xr-x. 1 jovyan users         24 Aug 18 20:25  parquet
time: 119 ms (started: 2026-09-04 12:35:00 +00:00)


In [21]:
csv_file = Path(path) / "male_players.csv"

time: 488 μs (started: 2026-09-04 12:35:00 +00:00)


In [22]:
%load_ext autotime

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 1.32 ms (started: 2026-09-04 12:35:00 +00:00)


In [23]:
%sql DROP TABLE IF EXISTS local.fifa.male_players_partition_fifa_update_date

Loading ITables v2.9.1 from the internet... (need help?)
🔒ⓘ


time: 101 ms (started: 2026-09-04 12:35:00 +00:00)


# Ingest partitioning by "fifa_update_date"

In [24]:
def ingest_table(table:str, partition: str):
    table = f"{table}_partition_{partition}"
    print(f"Table name: {table}")
    if spark.catalog.tableExists(table):
        print(f"Reading existing Iceberg table: {table}")
        return spark.table(table)

    print(f"Creating partitioned Iceberg table: {table}")

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(str(csv_file))
    )

    (
        df.writeTo(table)
        .using("iceberg")
        .partitionedBy(F.col(partition))
        .create()
    )

    return spark.table(table)
    

time: 456 μs (started: 2026-09-04 12:35:01 +00:00)


In [25]:
df = ingest_table("local.fifa.male_players", "fifa_update_date")

Table name: local.fifa.male_players_partition_fifa_update_date
Creating partitioned Iceberg table: 
local.fifa.male_players_partition_fifa_update_date
time: 37.5 s (started: 2026-09-04 12:35:01 +00:00)


# Number and size of files
 - Issue: 2MB small file size problem. Over-partitioning
 - Better size: 128MB to 512MB

In [26]:
%%sql
SELECT count(*) as count
FROM local.fifa.male_players_partition_fifa_update_date.data_files;


SELECT
    file_path,
    file_format,
    record_count,
    format_number(file_size_in_bytes, 0) AS file_size_in_bytes
FROM local.fifa.male_players_partition_fifa_update_date.data_files;

Loading ITables v2.9.1 from the internet... (need help?)
🔒ⓘcount
553


time: 231 ms (started: 2026-09-04 12:35:38 +00:00)


In [27]:
%%sql
SELECT *
FROM  local.fifa.male_players_partition_fifa_update_date
WHERE fifa_update_date="2014-11-28"
LIMIT 3


time: 135 ms (started: 2026-09-04 12:35:38 +00:00)


In [28]:
table = "local.fifa.male_players_partition_fifa_update_date"

time: 105 μs (started: 2026-09-04 12:35:38 +00:00)


In [29]:
df_select_filter = (
    spark.table(table)
    .filter(F.col("fifa_update_date") == "2014-11-28")
    .select("fifa_update_date", "club_name")
)

time: 15.8 ms (started: 2026-09-04 12:35:38 +00:00)


In [30]:
print(
    df_select_filter
    ._jdf
    .queryExecution()
    .executedPlan()
    .toString()
)

*(1) ColumnarToRow
+- BatchScan local.fifa.male_players_partition_fifa_update_date 
IcebergScan(table=local.fifa.male_players_partition_fifa_update_date, 
schemaId=0, snapshotId=5201370172287620672, branch=null, 
filters=fifa_update_date IS NOT NULL, fifa_update_date = 16402, runtimeFilters=,
groupedBy=) RuntimeFilters: []

time: 20.5 ms (started: 2026-09-04 12:35:38 +00:00)


In [31]:
all_files = spark.table(table).inputFiles()
filtered_files = (
    spark.table(table)
    .filter(F.col("fifa_update_date") == F.lit("2014-11-28"))
    .inputFiles()
)
print("All files:", len(all_files))
print("Filtered files:", len(filtered_files))

All files: 0
Filtered files: 0
time: 34.7 ms (started: 2026-09-04 12:35:39 +00:00)


# PushedFilters doesn't appear in the dataframe explain!
Partition pruning is the main optimization. The absence of the literal PushedFilters label does not mean the filter was ignored.

In [32]:
df_select_filter.explain("formatted")

== Physical Plan ==
* ColumnarToRow (2)
+- BatchScan local.fifa.male_players_partition_fifa_update_date (1)


(1) BatchScan local.fifa.male_players_partition_fifa_update_date
Output [2]: [fifa_update_date#2314, club_name#2330]
IcebergScan(table=local.fifa.male_players_partition_fifa_update_date, schemaId=0, snapshotId=5201370172287620672, branch=null, filters=fifa_update_date IS NOT NULL, fifa_update_date = 16402, runtimeFilters=, groupedBy=)

(2) ColumnarToRow [codegen id : 1]
Input [2]: [fifa_update_date#2314, club_name#2330]


time: 1.34 ms (started: 2026-09-04 12:35:39 +00:00)


In [33]:
df_select_filter.explain("extended")

== Parsed Logical Plan ==
'Project ['fifa_update_date, 'club_name]
+- Filter (fifa_update_date#2314 = cast(2014-11-28 as date))
   +- SubqueryAlias local.fifa.male_players_partition_fifa_update_date
      +- RelationV2[player_id#2310, player_url#2311, fifa_version#2312, fifa_update#2313, fifa_update_date#2314, short_name#2315, long_name#2316, player_positions#2317, overall#2318, potential#2319, value_eur#2320, wage_eur#2321, age#2322, dob#2323, height_cm#2324, weight_kg#2325, league_id#2326, league_name#2327, league_level#2328, club_team_id#2329, club_name#2330, club_position#2331, club_jersey_number#2332, club_loaned_from#2333, club_joined_date#2334, ... 85 more fields] local.fifa.male_players_partition_fifa_update_date

== Analyzed Logical Plan ==
fifa_update_date: date, club_name: string
Project [fifa_update_date#2314, club_name#2330]
+- Filter (fifa_update_date#2314 = cast(2014-11-28 as date))
   +- SubqueryAlias local.fifa.male_players_partition_fifa_update_date
      +- RelationV

---

In [34]:
viewdf(df_select_filter, limit=3)

time: 38.3 ms (started: 2026-09-04 12:35:39 +00:00)


# Table schema

In [35]:
df.printSchema()

root
 |-- player_id: integer (nullable = true)
 |-- player_url: string (nullable = true)
 |-- fifa_version: integer (nullable = true)
 |-- fifa_update: integer (nullable = true)
 |-- fifa_update_date: date (nullable = true)
 |-- short_name: string (nullable = true)
 |-- long_name: string (nullable = true)
 |-- player_positions: string (nullable = true)
 |-- overall: integer (nullable = true)
 |-- potential: integer (nullable = true)
 |-- value_eur: integer (nullable = true)
 |-- wage_eur: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- dob: date (nullable = true)
 |-- height_cm: integer (nullable = true)
 |-- weight_kg: integer (nullable = true)
 |-- league_id: integer (nullable = true)
 |-- league_name: string (nullable = true)
 |-- league_level: integer (nullable = true)
 |-- club_team_id: integer (nullable = true)
 |-- club_name: string (nullable = true)
 |-- club_position: string (nullable = true)
 |-- club_jersey_number: integer (nullable = true)
 |-- club_loane

# Finding the best partition collumn
Data distribution, cardinallity

In [ ]:
%%sql 
SELECT fifa_update_date, count(*) as count
FROM local.fifa.male_players 
GROUP BY fifa_update_date
ORDER BY count DESC;

In [ ]:
%%sql
WITH summary AS (
    SELECT
        COUNT(*) AS total_rows,
        COUNT(fifa_update_date) AS non_null_rows,
        COUNT(*) - COUNT(fifa_update_date) AS null_rows,
        COUNT(DISTINCT fifa_update_date) AS distinct_values
    FROM local.fifa.male_players
)
SELECT
    format_number(total_rows, 0) AS total_rows,
    format_number(non_null_rows, 0) AS non_null_rows,
    format_number(null_rows, 0) AS null_rows,
    format_number(distinct_values, 0) AS distinct_values,
    concat(
        format_number(distinct_values * 100.0 / total_rows, 4),
        '%'
    ) AS cardinality_pct,
    format_number(
        total_rows * 1.0 / NULLIF(distinct_values, 0),
        2
    ) AS average_rows_per_value
FROM summary;

In [ ]:
%%sql
SELECT club_team_id, count(*) as count
FROM local.fifa.male_players 
GROUP BY club_team_id
ORDER BY count DESC;

## Listing the table

In [ ]:
%%sql 
SELECT * 
FROM local.fifa.male_players 
LIMIT 3;

In [ ]:
%%sql
SELECT COUNT(*) AS data_file_count
FROM local.demo.people.data_files;


In [ ]:
%%sql
SELECT
    file_path,
    file_format,
    record_count,
    file_size_in_bytes
FROM local.demo.people.data_files;

In [ ]:
%%sql
-- To see the partition layout:
SELECT *
FROM local.demo.people.partitions;

In [ ]:
%%sql
-- To see which sort-order ID each data file uses:
SELECT
    file_path,
    sort_order_id
FROM local.demo.people.files;

In [ ]:
%run_nb spark-show

INFO:SparkMonitorKernel:Scala socket closed - empty data
INFO:SparkMonitorKernel:Socket Exiting Client Loop
INFO:SparkMonitorKernel:Starting socket thread, going to accept
